# Module 9 — CI/CD (GitHub Actions -> Databricks Jobs)
Exam domain: **Databricks Tooling**

This module is mostly about config files rather than Spark code, so it runs
fine in Colab: it generates a real `pytest` suite, a `databricks.yml` Asset
Bundle, and a GitHub Actions workflow YAML you can drop straight into a repo.

In [ ]:
!pip install -q pytest

## 1. The tests CI will run
Same tests from Module 8 — CI's job is to run exactly what you'd run locally,
automatically, on every push.

In [ ]:
import os
os.makedirs("/content/ci_demo/tests", exist_ok=True)

test_code = '''
def test_amount_must_be_positive():
    def clean(amount):
        return amount if amount > 0 else None
    assert clean(10) == 10
    assert clean(-5) is None
'''
with open("/content/ci_demo/tests/test_smoke.py", "w") as f:
    f.write(test_code)

!cd /content/ci_demo && python -m pytest tests/ -v

## 2. Databricks Asset Bundle (`databricks.yml`)
Bundles describe Jobs/DLT pipelines as code, so "deploy" means
`databricks bundle deploy`, not clicking through the UI.

In [ ]:
bundle_yaml = '''
bundle:
  name: databricks-de-portfolio

resources:
  jobs:
    daily_batch_pipeline:
      name: "daily-batch-pipeline"
      tasks:
        - task_key: run_module_3
          notebook_task:
            notebook_path: ../databricks/module_3_batch_pipelines_databricks.ipynb
            base_parameters:
              min_order_date: "{{job.trigger.time.iso_date}}"
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: "UTC"

targets:
  dev:
    mode: development
    workspace:
      host: https://<your-workspace>.cloud.databricks.com
  prod:
    mode: production
    workspace:
      host: https://<your-workspace>.cloud.databricks.com
'''
with open("/content/ci_demo/databricks.yml", "w") as f:
    f.write(bundle_yaml)
print(bundle_yaml)

## 3. GitHub Actions workflow
Lint -> test -> (on `main`) deploy the bundle to the `prod` target.

In [ ]:
workflow_yaml = '''
name: ci-cd

on:
  pull_request:
  push:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install pytest pyspark==3.5.1
      - run: python -m pytest tests/ -v

  deploy:
    needs: test
    if: github.ref == \'refs/heads/main\'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - run: databricks bundle deploy --target prod
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
'''
os.makedirs("/content/ci_demo/.github/workflows", exist_ok=True)
with open("/content/ci_demo/.github/workflows/ci-cd.yml", "w") as f:
    f.write(workflow_yaml)
print(workflow_yaml)

## What each stage is actually checking
- **test job**: runs on every PR *and* every push — catches breakage before
  it merges.
- **deploy job**: gated by `needs: test` (won't run if tests fail) and
  `if: github.ref == 'refs/heads/main'` (only deploys from the trunk branch,
  never from a PR branch).
- Secrets (`DATABRICKS_HOST`, `DATABRICKS_TOKEN`) come from GitHub repo
  secrets, never hardcoded — a service principal token, not a personal PAT, in
  a real production setup.